In [1]:
%load_ext autoreload
%autoreload 2

# 4. Seasonal Naive T-7 Baseline Training

Train baseline naive model.

In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from src.data.loader import load_features
from src.models.splitting import split_train_holdout, split_features_target

from src.models.seasonal_naive_model import SeasonalNaiveModel

from src.utils.helpers import save_pickle


## Configure experiment

In [3]:
feature_path = Path('../data/features')
artifact_dir = Path('../experiments/exp_000_seasonal_naive/artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)

store_filter = ['CA_1', 'TX_1']
split_test_date = pd.Timestamp('2016-04-24') #-28 days from the last date in the dataset
split_validation_date = pd.Timestamp('2016-03-27') #-28 days from the test date
target_col = 'sales'
drop_cols = ['id', 'date', target_col]


## Load data

In [4]:
print(f'Loading features from: {feature_path}')
df = load_features(feature_path, store_filter=store_filter)

# Holdout data for testing and training/validation 
train_df, test_df = split_train_holdout(df, split_date=split_test_date)

# Prepare data for training and validation
X_train, y_train = split_features_target(train_df, target_col=target_col, drop_cols=drop_cols)

# Prepare data for testing
X_test, y_test = split_features_target(test_df, target_col=target_col, drop_cols=drop_cols)

Loading features from: ..\data\features
Applied store filter: ['CA_1', 'TX_1']


## Run experiment

In [5]:
# Train and evaluate a naive model
naive_model = SeasonalNaiveModel(name='naive_seasonal_t7_exp001')
naive_model.train(X_train, y_train)

naive_prediction_parts = []
naive_sales_history = train_df[['id', 'date', target_col]].copy()
naive_sales_history = naive_sales_history.rename(columns={target_col: 'sales'})

for current_date in sorted(test_df['date'].unique()):
    current_mask = test_df['date'] == current_date
    current_test = test_df.loc[current_mask].copy()
    X_current = X_test.loc[current_mask].copy()

    history = naive_sales_history.set_index(['id', 'date'])['sales']
    lag_date = current_date - pd.Timedelta(days=7)

    X_current['sales_lag_7'] = [
        history.get((item_id, lag_date), np.nan)
        for item_id in current_test['id']
    ]

    y_hat = naive_model.predict(X_current)

    current_predictions = current_test[['id', 'date', target_col]].copy()
    current_predictions = current_predictions.rename(columns={target_col: 'y_true'})
    current_predictions['y_pred'] = y_hat
    naive_prediction_parts.append(current_predictions)

    predicted_sales = current_test[['id', 'date']].copy()
    predicted_sales['sales'] = y_hat
    naive_sales_history = pd.concat([naive_sales_history, predicted_sales], ignore_index=True)

naive_predictions_df = pd.concat(naive_prediction_parts, ignore_index=True)

print(f'Naive predictions: {len(naive_predictions_df):,} rows')
display(naive_predictions_df.head(10))

Training naive_seasonal_t7_exp001 with 11665474 samples and 22 features...
[+] Seasonal naive t-7 ready using 'sales_lag_7'
Naive predictions: 170,744 rows


,id,date,y_true,y_pred
0,FOODS_1_001_CA_1,2016-04-25,2,4.0
1,FOODS_1_001_TX_1,2016-04-25,0,0.0
2,FOODS_1_002_CA_1,2016-04-25,0,0.0
3,FOODS_1_002_TX_1,2016-04-25,6,0.0
4,FOODS_1_003_CA_1,2016-04-25,3,1.0
5,FOODS_1_003_TX_1,2016-04-25,0,1.0
6,FOODS_1_004_CA_1,2016-04-25,0,0.0
7,FOODS_1_004_TX_1,2016-04-25,0,0.0
8,FOODS_1_005_CA_1,2016-04-25,1,2.0
9,FOODS_1_005_TX_1,2016-04-25,2,0.0


## Save Results

In [6]:
# Save model and results to experiment artifacts
artifact_dir.mkdir(parents=True, exist_ok=True)

if isinstance(store_filter, (list, tuple, set, pd.Index)):
    store_suffix = '_'.join(map(str, store_filter))
else:
    store_suffix = store_filter if store_filter else 'all_stores'
    
# Save train_df for future reference
train_df_path = artifact_dir / f'train_df_{store_suffix}.parquet'
train_df.to_parquet(train_df_path, index=False)
print(f'✓ train_df saved to {train_df_path}')

# Save model
model_path = artifact_dir / f'seasonal_naive_{store_suffix}.pkl'
save_pickle(naive_model.get_model(), model_path)

# Save predictions
results_path = artifact_dir / f'predictions_{store_suffix}.parquet'
naive_predictions_df.to_parquet(results_path, index=False)
print(f'✓ Predictions saved to {results_path}')

✓ train_df saved to ..\experiments\exp_000_seasonal_naive\artifacts\train_df_CA_1_TX_1.parquet
Saved to ..\experiments\exp_000_seasonal_naive\artifacts\seasonal_naive_CA_1_TX_1.pkl
✓ Predictions saved to ..\experiments\exp_000_seasonal_naive\artifacts\predictions_CA_1_TX_1.parquet
